# Week 11 과제 · 데이터 정제 · 변환 · 집계

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MNU-AI-Programming/AI-Assisted-Python-Programming/blob/main/weeks/week11/assignment.ipynb)

## 제출 방법

1. **파일 → 드라이브에 사본 저장** 후 파일 이름을 `week11_학번_이름.ipynb`로 바꿉니다.
2. `# TODO` 부분을 채우고 각 문제 아래 **자동 확인 셀**이 ✅인지 확인합니다.
3. 문제별 **AI 활용 기록**과 마지막 **성찰**을 작성합니다.
4. **런타임 → 다시 시작 및 모두 실행** 후 `.ipynb` 파일을 제출합니다.

> **규칙**: 모든 함수는 입력을 바꾸지 않고 새 Series/DataFrame을 돌려줍니다.

| 문제 | 내용 | 배점 |
|---|---|---:|
| 1 | 학과명 표준화 | 2 |
| 2 | 코딩 시간 숫자 변환과 이상값 | 2 |
| 3 | 설문 정제 파이프라인 | 3 |
| 4 | 단과대학 요약과 피벗 (AI 협업) | 3 |
| | **합계** | **10** |

## 준비: 데이터와 자동 확인 도우미

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/MNU-AI-Programming/AI-Assisted-Python-Programming/main/datasets/"

def get_data(name):
    for folder in [Path("../../datasets"), Path("datasets"), Path(".")]:
        if (folder / name).exists():
            return str(folder / name)
    urllib.request.urlretrieve(DATA_URL + name, name)
    return name

RAW = pd.read_csv(get_data("survey_raw.csv"), dtype=str, keep_default_na=False)
DEPTS = pd.read_csv(get_data("departments.csv"))
RAW_BACKUP = RAW.copy()
STANDARD = ["간호학과", "경영학과", "전자공학과", "정보보안학과", "컴퓨터공학과"]


def check(desc, test):
    """자동 확인 도우미: test는 True/False를 돌려주는 함수(lambda)입니다."""
    try:
        ok = test()
    except NotImplementedError:
        print(f"⏳ 미작성 - {desc}")
    except Exception as e:
        print(f"❌ 오류 - {desc} ({type(e).__name__}: {e})")
    else:
        print(f"{'✅ 통과' if ok else '❌ 실패'} - {desc}")

---
## 문제 1. 학과명 표준화 (2점)

`normalize_dept(s)`는 학과명 Series를 받아 표준 학과명 Series를 돌려줍니다.

1. 앞뒤 공백과 **중간 공백을 모두** 제거합니다.
2. 줄임말을 표준 이름으로 바꿉니다: `컴공`, `정보보안`, `보안학과`, `전자공`, `경영`, `간호`
3. 위 과정 후에도 **표준 5개 학과가 아닌 값은 `"기타"`** 로 바꿉니다. (빈 문자열 포함)

In [ ]:
def normalize_dept(s):
    # TODO: 여기에 코드를 작성하세요
    raise NotImplementedError("코드를 작성하세요")

In [ ]:
sample = pd.Series([" 컴퓨터공학과", "컴퓨터 공학과", "컴공", "보안학과", "정보보안", "전자공학과 ", "경영", "간호", "물리학과", ""])
check("표준화 결과", lambda: normalize_dept(sample).tolist() == ["컴퓨터공학과", "컴퓨터공학과", "컴퓨터공학과", "정보보안학과",
      "정보보안학과", "전자공학과", "경영학과", "간호학과", "기타", "기타"])
check("설문 전체: 표준 5개만", lambda: set(normalize_dept(RAW["department"])) == set(STANDARD))
check("원본 Series 불변", lambda: (normalize_dept(sample), sample.iloc[0] == " 컴퓨터공학과")[1])

---
## 문제 2. 코딩 시간 숫자 변환과 이상값 (2점)

`parse_hours(s, max_hours=16)`는 `"2시간"`, `"약 2.5"`, `"3"`, `""`, `"30"`, `"-1"` 같은 문자열 Series를 실수 Series로 바꿉니다.

- 문자열에서 **첫 번째 숫자**(음수·소수 포함)를 꺼냅니다.
- 숫자가 없으면 `NaN`
- `0` 이상 `max_hours` 이하가 아니면 `NaN` (자르지 않음)

In [ ]:
def parse_hours(s, max_hours=16):
    # TODO: 여기에 코드를 작성하세요
    raise NotImplementedError("코드를 작성하세요")

In [ ]:
hs = pd.Series(["2시간", "약 2.5", "3", "", "30", "-1", "0", "16", "1.5시간", "많이"])
expect = [2.0, 2.5, 3.0, np.nan, np.nan, np.nan, 0.0, 16.0, 1.5, np.nan]
check("변환 결과", lambda: np.allclose(parse_hours(hs).to_numpy(dtype=float), expect, equal_nan=True))
check("max_hours 매개변수", lambda: parse_hours(hs, max_hours=2).notna().sum() == 3)
check("설문 전체 결측 수 (빈 칸 + 범위 밖)", lambda: parse_hours(RAW["coding_hours"]).isna().sum() == 15)
check("자료형 float", lambda: parse_hours(hs).dtype.kind == "f")

---
## 문제 3. 설문 정제 파이프라인 (3점)

`clean_survey(raw)`는 원본 설문을 다음 열을 가진 DataFrame으로 바꿉니다. 문제 1, 2의 함수를 재사용하세요.

| 열 | 정제 규칙 | 자료형 |
|---|---|---|
| `response_id` | 공백 제거 | 문자열 |
| `submitted_at` | `format="mixed"`로 날짜·시각 변환 | datetime |
| `weekday` | 제출 요일 `"월"`~`"일"` (`.dt.dayofweek` 활용) | 문자열 |
| `department` | 문제 1 | 문자열 |
| `grade` | 숫자 한 글자 추출 | 정수 |
| `coding_hours` | 문제 2 | 실수 |
| `uses_ai` | 예/Y/yes/사용함 → `True`, 아니오/N/no/사용안함 → `False` | bool |
| `satisfaction` | 숫자 추출, 결측은 그대로 | `Int64` |

- 순서: **표기 통일(정제) → 중복 제거** (정제 후 모든 열이 같은 행은 하나만 남김)
- 결과는 `response_id` 순으로 정렬하고 인덱스는 0부터

In [ ]:
def clean_survey(raw):
    # TODO: 여기에 코드를 작성하세요
    raise NotImplementedError("코드를 작성하세요")

In [ ]:
EXPECTED_FIRST_WEEKDAY, EXPECTED_AI_RATE = "월", 0.793        # survey_raw.csv 기준값
check("열 구성", lambda: list(clean_survey(RAW).columns) == ["response_id", "submitted_at", "weekday", "department",
      "grade", "coding_hours", "uses_ai", "satisfaction"])
check("중복 제거 후 150행, ID 유일", lambda: len(clean_survey(RAW)) == 150 and clean_survey(RAW)["response_id"].is_unique)
check("자료형", lambda: (lambda c: c["submitted_at"].dtype.kind == "M" and c["grade"].dtype.kind == "i"
      and c["uses_ai"].dtype == bool and str(c["satisfaction"].dtype) == "Int64")(clean_survey(RAW)))
check("요일", lambda: set(clean_survey(RAW)["weekday"]) <= set("월화수목금토일")
      and clean_survey(RAW).loc[0, "weekday"] == EXPECTED_FIRST_WEEKDAY)
check("AI 사용 비율", lambda: round(clean_survey(RAW)["uses_ai"].mean(), 3) == EXPECTED_AI_RATE)
check("만족도 결측 4", lambda: clean_survey(RAW)["satisfaction"].isna().sum() == 4)
check("원본 불변", lambda: (clean_survey(RAW), RAW.equals(RAW_BACKUP))[1])

dup_after_clean = pd.DataFrame({"response_id": ["A1", "A1 "], "submitted_at": ["2025-09-01 10:00", "2025/09/01 10:00"],
                                "department": ["컴공", "컴퓨터공학과"], "grade": ["2", "2학년"], "coding_hours": ["2", "2시간"],
                                "uses_ai": ["Y", "예"], "satisfaction": ["4", "4점"], "comment": ["", ""]})
check("정제 후에 같아지는 중복도 제거 (순서)", lambda: len(clean_survey(dup_after_clean)) == 1)

---
## 문제 4. 단과대학 요약과 피벗 (AI 협업, 3점)

정제된 설문(`clean_survey(RAW)`)과 `departments.csv`를 이용해 두 함수를 작성하세요.

1. `college_summary(clean, depts)`: 단과대학(`college`)을 인덱스(가나다순)로 하고 다음 열을 가진 DataFrame
   - `responses`: 응답 수, `ai_rate`: AI 사용 비율(%, 소수 첫째 자리), `avg_hours`: 평균 코딩 시간(소수 둘째 자리, 결측 제외)
   - `merge` 후 **행 수가 변하지 않았는지** 함수 안에서 확인하고, 변했으면 `ValueError`를 냅니다.
2. `sat_pivot(clean)`: 행 = 학년(1~4), 열 = `"AI 미사용"`, `"AI 사용"`, 값 = 평균 만족도(소수 둘째 자리)

**AI 협업 과제**: AI의 첫 답에서 ① `merge` 후 행 수 확인이 있었는지, ② 만족도 결측을 임의로 채우지 않았는지 확인하세요.

In [ ]:
def college_summary(clean, depts):
    # TODO: 여기에 코드를 작성하세요
    raise NotImplementedError("코드를 작성하세요")


def sat_pivot(clean):
    # TODO: 여기에 코드를 작성하세요
    raise NotImplementedError("코드를 작성하세요")

In [ ]:
def _raises(exc, fn, *args):
    try:
        fn(*args)
    except exc:
        return True
    return False


CLEAN = None
try:
    CLEAN = clean_survey(RAW)
except NotImplementedError:
    pass

EXPECTED_ENG = [87, 80.5, 2.01]              # survey_raw.csv 기준값
EXPECTED_PIVOT_G3 = [2.43, 3.56]

check("단과대학 인덱스", lambda: list(college_summary(CLEAN, DEPTS).index) == ["간호대학", "경영대학", "공과대학"])
check("열 구성", lambda: list(college_summary(CLEAN, DEPTS).columns) == ["responses", "ai_rate", "avg_hours"])
check("응답 수 합계 150", lambda: college_summary(CLEAN, DEPTS)["responses"].sum() == 150)
check("공과대학 행", lambda: college_summary(CLEAN, DEPTS).loc["공과대학"].tolist() == EXPECTED_ENG)
bad_depts = pd.concat([DEPTS, DEPTS.iloc[[0]]])                       # 키가 중복된 표
check("merge로 행이 늘면 ValueError", lambda: _raises(ValueError, college_summary, CLEAN, bad_depts))
check("피벗 모양 4×2", lambda: sat_pivot(CLEAN).shape == (4, 2) and list(sat_pivot(CLEAN).columns) == ["AI 미사용", "AI 사용"])
check("피벗 값", lambda: sat_pivot(CLEAN).loc[3].tolist() == EXPECTED_PIVOT_G3)

### AI 활용 기록 (문제 4)

| 항목 | 내용 |
|---|---|
| 사용한 AI 도구 |  |
| AI 첫 답에 merge 후 행 수 확인이 있었나? |  |
| 결측을 채우거나 이상값을 자르는 등 요청하지 않은 처리가 있었나? |  |
| 내가 직접 고친 부분과 이유 |  |

---
## 성찰 (필수)

1. 정제 규칙 중 "결측으로 둘지, 자를지, 채울지"를 결정할 때 무엇을 기준으로 삼았나요?
2. 정제 순서를 바꾸면 결과가 달라지는 예를 직접 만들어 보았나요?
3. merge 후 행 수 확인처럼, 앞으로 꼭 넣을 "안전 장치"는 무엇인가요?

(여기에 작성)